# HVAC Takeoff — GPU Batch Inference (Kaggle)

Runs the **full takeoff pipeline end-to-end** on every project in a corpus, on a Kaggle **T4 GPU** — dramatically faster than local CPU (YOLO + EasyOCR both move to GPU). Produces a consolidated report + per-project outputs you can download.

## Before you run
1. **Settings → Accelerator → GPU T4 x2** (or P100).
2. **Settings → Internet → On** (needed to clone the repo + download the EasyOCR model on first use).
3. **Add your corpus as a Dataset** (right panel → *Add Input*). Upload the `data to train` folder (or any folder of project subfolders) as a Kaggle Dataset first. It will mount under `/kaggle/input/<your-dataset>/`.

## ⚠️ Honest caveat — read this
If the corpus you attach is the model's **training data** (the 158-project `data to train` set), the resulting scores measure **memorization, not real accuracy** — the model has already seen these plans. This run is a valid **end-to-end robustness test** (does it run, not crash, produce a takeoff on every project?) and a **floor check** (if it scores poorly even here, that's a red flag). It is **not** a real accuracy benchmark — for that, use held-out plans the model never trained on.

## 1. Confirm the GPU is live

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY — fix Accelerator setting!')
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 2. Clone the repo (code + committed models) and install deps

In [ ]:
import os, subprocess
REPO_DIR = '/kaggle/working/hvac-takeoff-tool'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/triunesolutions/hvac-takeoff-tool {REPO_DIR}
else:
    print('repo already cloned; pulling latest')
    !cd {REPO_DIR} && git pull --ff-only

# Verify the production model came down with the clone (not Git-LFS-stubbed)
mp = os.path.join(REPO_DIR, 'models', 'hvac_yolov8s_v10.pt')
sz = os.path.getsize(mp) if os.path.exists(mp) else 0
print(f'model: {mp}  ({sz/1e6:.1f} MB)')
assert sz > 1e6, 'Model missing or LFS-stub. If LFS: run `git lfs install && git lfs pull` in the repo.'

In [ ]:
# Kaggle ships torch+CUDA already; add the pipeline deps.
!pip -q install ultralytics easyocr pdfplumber PyMuPDF openpyxl opencv-python-headless pandas

## 3. Point at your corpus

Auto-detects a `projects/` folder under `/kaggle/input/`. If your layout differs, set `CORPUS` by hand to the folder that contains the **project subfolders**.

In [ ]:
import glob
cands = (glob.glob('/kaggle/input/*/projects') + glob.glob('/kaggle/input/*/*/projects')
         + glob.glob('/kaggle/input/*/data*train*/projects'))
if cands:
    CORPUS = cands[0]
else:
    roots = sorted(glob.glob('/kaggle/input/*'))
    CORPUS = roots[0] if roots else None
print('CORPUS =', CORPUS)
assert CORPUS and os.path.isdir(CORPUS), 'Set CORPUS to your project-subfolders directory.'
n_proj = len([d for d in os.listdir(CORPUS) if os.path.isdir(os.path.join(CORPUS, d))])
print(f'{n_proj} project subfolder(s) found')

## 4. Run the batch (GPU)

`--recursive` picks one plan PDF per project (logged to `plan_selection.csv`), smallest first. `TIME_BUDGET = 0` means no per-plan cap — feasible on GPU. Set it to e.g. `300` as a safety net if a pathological plan stalls.

In [ ]:
import sys, subprocess
OUT = '/kaggle/working/batch_out'
TIME_BUDGET = 0          # GPU is fast; 0 = no cap. Use 300 as a safety net if needed.
cmd = [sys.executable, 'scripts/batch_run_all.py',
       '--input', CORPUS, '--out', OUT, '--recursive',
       '--time-budget', str(TIME_BUDGET)]
print(' '.join(cmd))
# Stream output live to the cell.
proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('\n=== batch exit code:', proc.returncode, '===')

## 5. Review the report + plan selection

In [ ]:
import pandas as pd
print('=== which PDF was picked per project (audit me!) ===')
sel = pd.read_csv(os.path.join(OUT, 'plan_selection.csv'))
display(sel.head(20))
print('\n=== results ===')
res = pd.read_csv(os.path.join(OUT, 'batch_results.csv'))
display(res)
print('\n=== report ===')
print(open(os.path.join(OUT, 'batch_report.md'), encoding='utf-8').read())

## 6. Bundle outputs for download

Zips the report, CSVs, and every project's `xlsx` + JSON sidecars (skips the big annotated PDFs and run logs to keep the download small). Find `batch_report_bundle.zip` in the right-panel *Output*.

In [ ]:
import zipfile
zpath = '/kaggle/working/batch_report_bundle.zip'
skip = ('_annotated.pdf', '_run.log')
n = 0
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT):
        for fn in files:
            if fn.endswith(skip):
                continue
            fp = os.path.join(root, fn)
            z.write(fp, os.path.relpath(fp, OUT))
            n += 1
print(f'Bundled {n} file(s) → {zpath}  ({os.path.getsize(zpath)/1e6:.1f} MB)')